# 🕵️ Dia 58 — Do armazenamento Plaso a uma história investigativa no Timesketch

**Investigador Cyber 90 Dias** · Fase 4 · Rank Jutsu `B`

> Baseado no *Day 58: From Plaso storage to a Timesketch investigation story* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-4-digital-forensics-incident-investigation/day-58.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

A Marta quer saber, em ordem, o que aconteceu depois que a caixa abriu o link no `SA-CX-07`. Rafa avisa: *"Ordem cronológica não é história. História é: este evento está ligado àquele **por causa deste campo**."*

Neste notebook você vai:

1. **Montar a linha do tempo "fw+app" do LAB-P4** já corrigida e exportá-la no CSV que o Timesketch aceita.
2. **Rodar um mini-Timesketch**: buscar, aplicar *tags*, salvar buscas.
3. **Achar as ligações entre fontes pelo campo compartilhado** (porta 41766, IP 198.51.100.23).
4. **Escrever a história** com setas cheias (campo) e tracejadas (inferência).
5. **Fazer o mesmo no `SA-CX-07`** e jogar o quiz "seta cheia ou tracejada?".

🐍 Só biblioteca padrão. O Timesketch de verdade precisa de uma VM com 8 GB; aqui você entende a lógica.

## 1. Carregando as evidências

In [ ]:
# ------------------------------------------------------------
# CARREGADOR OFFLINE-FIRST
# Tenta baixar cada arquivo do GitHub (raw). Se a rede falhar,
# usa a cópia EMBUTIDA aqui embaixo (comprimida para caber no notebook).
# Nos dois casos, calculamos o SHA-256: é a nossa "cadeia de custódia" de bolso.
# ------------------------------------------------------------
import base64          # para decodificar as cópias embutidas
import hashlib         # SHA-256 de cada arquivo carregado
import urllib.request  # download simples, sem bibliotecas externas
import zlib            # as cópias embutidas estão comprimidas com zlib

# Endereço "cru" dos dados do LAB-P4 no repositório original (ThiefDroppers)
RAW_LAB = ("https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/"
           "curriculum/phase-4-digital-forensics-incident-investigation/resources/case-lab-p4/")
# Endereço "cru" dos dados sintéticos da Operação Isca neste repositório
RAW_BR = ("https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/"
          "caso-continuo/dados/")

# Plano B: cópias fiéis dos arquivos, feitas a partir do repositório local.
EMBUTIDOS = {
    "lab:logs/bastion01-auth.log": """
eNq1m01v20YQhu/5FTzGhxo7s/OxS6AHA23RS3rppUBQBPpyIkSWVFFK4n9fWi7S2HztOuQ0EBLHMPRwd/nuzD603swODeWGpE3c
EjXzWXdc77aJmq77sHzLVJL/2TZXi8Vqf1wtm/1pvlkvPq5um+vdodntu8vZ8ma9ba4Pu5uG0mX/0v5vbva7w7FRUpG7d+K2+fkn
VqXa/P7rFau1f5Ub+uNL2szmqbvdHj+sjutF6t83Xa+371eH/WG9Paar/s+rNw8ukZ+6xP3s5t1pu/7y+u67bbfquv6HLtrmn6/6
a11t+wHcXfapW31z7a9P6+WPlBJdNPPb5vy/dPGVaq14myugkkyYGE6UoibmfImSnrrE/2NipOnXQqlNGVG1p/4yW2/upmXWdZ93
h/u3X28/zTbr5T3mm+nhlC/7uaF8KXo/P1I5lfP8POSRIp49wTvsdsdnCP1iDAksiOAvGdF8tvh42j8HNADMEFheAtwdZovN6hkg
CwAKvJnrE8Du0+Ldfw4spwGH24QGxuklAzuuuucWLjvAUUE4egnuND9tj6dngP2XQ2BGiWN+CfD6uL//92liRUS0cpwj0qYMeIo2
W5ZxadPhzZ/7HR0RNCRtJgDYRxAALSRtVgCQ4Zr5lLRZBRyBM1kC0uZoWIpaBa4hafPh+ARXmpxi0lYcEOGGkikibdUAL6MVzDwq
bbmfriFB4BzmiLRlIgBUCJSItGVQTBXXnKwT0paZAAduI9mmpy2DUqoto7Yne0TaMiim2vbfBMASkrYsGRAVVdNcA9KWZThCw5VH
0ri0SQEEQveiUEjaFA2J0aIJh6RNKwDCmiN5StoMzaTCmZSAtPmwR/A2oY1fNCRtbgDIcB4tJm2gmjo+dIhHpK1kwBO0gUkZlzZQ
rb1V1GdJDUlbHbb/BcsSTSFpqwUAGdUcpQlpE9AllDaj2195etqEEsDB46jmiLQJaEoq7iRVQtImVAERVgKNsCTCDHgMV3CcJRHQ
kFTcSWqIJZEsAAhLgIZYEsmP9uT+WJraBG/LKZZEJAEO9E0WYEnkcZt1xkGFYCGWRB73WWegoHvRYiyJ6HBGCXd2FmFJRAvgEWq7
bJwlESNAgA7BQiyJOJpCfJeEWBJxNIfwoGhTLImUYaoZt3YWYEmkEsAxHFaIJZFaATCjHsFjLIkmAUQogjzCkigNpzTj6u3jLIk+
fgJwJhAkhFgSJQdAWK49xJIoCwDCZ0U+xZIoK+Ao6vo9wJJoHg5L2gRxIZZEcwVAuPN7jCVRUE2lzTBtEZZE1QBP0H5SxlkSBbVN
+xYBEUIsiZoCIGzGS4glUSsACItAmWJJ1A1woD8uAZZE3QFO4DyGWBItDIBwNykxlkTLMN/WvxAxwpJozYDHKN1lnCXRWgABep8S
YkksMQDC9r+GWBJLCoCK9uQ6xZIYDYuN9y/ECbAkBnoExw+jaoglMVC7HT/drjGWxHJGRHToqBGWxMAJ33HnWsdZEgNn+9IfbBAh
xJIYaA9KS3DRQiyJSUVAuGZTLIkpAQ5qkWsKsCSmCnDowUZNIZbEFC0c0ls1xVgSA9YCW9CaIiyJaQU8giMcZ0nMBBDQQ9KaQiyJ
gU6ywoekNYVYEvMMgGjHqmmKJbEySBsl2JPXFGBJbNhm9TiCuBBLYtUQEOxalWIsiacMiBmljSIsiQ8dRs8TdGvSOEviNBwRtqyV
QiyJcwJAgsAQS+LsAMgQOMWS+FBb9Bzk7CoFWBIXBjiBN3+IJXEZjg//HmilGEviooBIMG0RlsSH3QlhzVt5nCVxzYAASw2HWBJX
NIVwA+EQS+LD/oAy7JUrT7Ekbglw0G+QVw6wJG4GcOjxXuUQS+IuAAh3E46xJF6G96bAk3Dlh58a+b6100TMiFSeIn3vZxz+vYb7
DzlQcvghB4C97Y6rm+UPm9379Xb51s6/X/Xb6vNXVO6P6rvrx5jLb95SGcqzl41ksdl1eCSv/gayfscQ""",
    "lab:logs/fs01-auth.log": """
eNqdkVtPGzEQhd/5FfNIHog93ksSozxsC1Jf0guhKm1VoV2vl1hkbWN7I8KvrzekZBVRCTGyrLE1Oufz8aJ0gAnQGcecswwaTxG8
X9W/EafTPxwKIaQNsgbbVWsl7uUWGuPAWD8u61ZpaJxpAek4rizuDKxxATI6i20UYhwuL1iW4QyWnwqW5fxh2uLNI12XFfVbHVYy
KEGjLm2UvpPOOqUDLWKdLHq4FGjCGeXpEC6hKeL/6fxG3FaluO/sEI9Fwme6FCd5vqe7Whb/yH5VkwPRjm8jnmV2fB9e6ogsf4XM
lu1tp9XjaX/LvfReGT3isO9igFJH7p6283KIfNqpeo5IJyOotrA70dHAMUk4TfaOXW2iY0ZZdBw8msP19c+5DZ4gnMPXHxdzsjKt
JIORc/i+vLyaO2NC7D9+WSyKz3Gs845USpMQ7c7EUwNkUzoSWkvGDzgOd09AvNuQ2og+yHJN4qeVWkjCKMvPvuEbOA/JxNs3JNMj
7lPoA3ktqKFrytn0Ha5ibfyx60A2Q87wHd98LHugP/kLLqYZ1g==""",
    "lab:logs/firewall.log": """
eNqt291qXDcQB/D7PMW+wC6ab+lACsXpdU1JH8AhDgkYu2Q3hLx9tc42tY4d/7XxgMlV7N9KGs3M0VlxYd8W2ZJsyJcSSymbD18L
bX6/uPjj8u3m7cXlhsqu/2jZWSzaWuXN9rcNNd6VHe+4LKqy+fzl5vr11c3N3dftx8Phn/327sth8+7b4Xr/Ws21veIBovZzyPq/
vFgpOkAEIaJmvoK0QKi1Jg8hmYBUaQWZ4qlz8gFyBEVVlzVUn4WcFjOqA8QQEtE6OLGwIEdFw4Ylqsjh2lqsoYBLVCvTMHMEoRJq
K6jPNxqRxhjdOBZCi60hZTSivo1iGJFCiIN1cOpC2PEqet52leiLtIYUxlxtZYgFVjigHttryGAsKEkbgpscQ/3DDVBbJOCIgric
F3SVYpUXWt8kMLqjthimLuAa0ZhR+6oSTN3dqXUYUMPbtVpZQYx3UfgI4SVS6yl1BU3sIqZVLMCZi2J1HBF15Qf05q8/L78zre6M
OlV20WPBezq5h+517h9gEantu/T++sPVl5vD9v317bcT448MftbwHvpk9kID79LWdFwZgStDEraGHFdv8jGRMs471WQFPZfgTkXV
Y6xBgvNOUaprKJ5dnurHjH2qQb+6PAyMvjzMp3j+VUNgX9XLlQ9pjWEWUG1jWuuQEgwBqirn5U/XXujXkMF0Y1zKeSFgscqffe4J
1+waet7mYVFex7TBzaPVx1oqBjvSvneG1pcZFe1jRSh9jc6D3IJtBSmGqseYqHHpIeayhhoekfGQ4NjwGgmtoWMw8COIi/S/SSS7
HpTauNTVXmV+CO33H7fvrvaHT3e3P3IpPyGRQok4RWLBkqdIgiXWFKnXMyRJSZB6JcJjkkiRKKCkliJxw1JLkQSvk3GKZISljCjv
jT2WXFMkMizVFInxOnlLkRTPXuSMyQqWMsakM1WjRoo0kSOap0gC10mKpEgKZ0+IUiSbkDJmzyaqhjClSDhHSEol7L+lWIoUSRxK
KimSNixljMknqoZoTZEIx57ljInxOllLkXDVEM+ZPcOzFxnVPfB5Wpif98herY4H+ico8JA8ZUiMlyml5MbEo4ZUSZEUp72U4n58
XoZSy3gAqAsVLNUUiWF50pQ2oi4Co1yppEj44VNTGpY20VgqtRQJFw1lTpEYr1NKw9ImGksVTZFw0VB5cYaV+1cWOPa0pEj42Ehf
3oTdSzIxpkiRFMeeZcweTbR7ajVFItiEqVOKhA8JNHJmbyIiImf28COh1oydyxNdmDZKkRiPqbUUSWB1t6IpEj7MMcqYPZmoufby
g/l7iSakSJFwzTXWFEkalixFMtiXm2SMSfsPllqKhHO5pVRCXQTvJ/MUSXGOSKlP2hsxLGXEnk10y+Y1RcK53MJTJHzgaxEpkuLZ
q5wiTeSImrFzvWNQapIiMd65raZI+IzFC6dIuC//77sRL5UMZlinjKoRHcNSpEj4pZCn1NyYeHHsIikSfgLwlCfqmOgsPeWJ+skv
cD+SIkfC66QtRSK8TkYpEu5h3SxFwi8b3HLWCZ8becopwcxJoltLkXBv5K4pEn4h6Sn9Xpt4IekhKdJENqqUI+EckdEbUZnJRs1T
JJyNokiKJHA/RcYpwfH6k2MpY0wzZ5bBJUWiCSlSJPxGMjJOCbqEz8JCOUXC759CM8Y08+XHUEuR8FlYZPQRNHM6GiYpEq4aYTmz
N5EjMqo7yUQPG15SJPwF6XBPkfCbrghNkSZyRM2IPYVPn1aIM7JReyrDPvibSsMqSTne1noMfbo9XH++vbr5cbGhri6GycM7vT+7
S8Xt0RvJcy7SyBP3W/5+czl8dJPhPXhXjk38g9G8v92vR9P4CYZ/MmvfGWl2Oob4f4SMb21VJmllPXP2/My10v9HvOgKktx/5xHc
b7F66ienb4OICbdX/wLJa96W""",
    "lab:logs/docportal-app.jsonl": """
eNrF1kFPgzAUAOC7v2LhLOW9thTGzYuJlxmNiYmXhXXdRtYBgYIa43+3nYlRs0nNogQO9LWkHy+PBy+BaYNsElCgIgQWIrtDllGa
UfYQnE8CrXql3YKr2eW1C9hxaVxgWcluZ69JX6hHN9O1qnETuZSGSK2aLaCLt42cF7WbQSD25EAEcxN1bjYuHK2KMi+livaIG4yK
sg9RJCmplyu3UFbdfk98PXs56I3ZqV6t8iXQI9w4GeamTHhyeUbFCVy7O9mtm5Nyy6fMH5v8JRaHsQjgjeXiTwvBJ7cM+D9yf37P
fApXpL61EGdARy5cjP2xTIxcuCn1xYoMcWQsx6k/duz+BZR7YxmM3WxT9MdOR+9fIhXeXD72t4EL39wmGcDYWOb1R8PvwP7O8IwN
YfPObIiu1kX5Wdr2cr7I5barv1FpQsAee+hOmU21dGFdyVyHptqq0g8W2171FXZ/cTs7nMVFp7dz9VRXjfk18VguPys5UjtaPBvl
pDwFEIj0BzqNB3L6jiWy2tVaGXVc/cFrmz6yD+xuy/UA9vXsDceHGjg=""",
    "br:dia-57/proxy-sa.log": """
eNq9Wk1v20YQvedX8A9I3vnYLwI+BUFu8SW3ogdWUlOhsqRIFODm13e4ayukIHI3rLeADjYEcx7fzrx5M+unVVtVugJVq+5THU+H
l38W56Y6f79s178hAvxey9dLVEtWSw3V6mW52uzb00Hp6uPTly+fPn6tnptjc15uXprn425TM1OF8qzDpX1EQ7ra7h+ZvaVqfW4f
weNSHrcE+PD0M7iRTzI4Vt82p7fokBGdHLsQ3VjtBtHJ96J3H0xGp96rm2vwrfze7DftYrXe38UA1voOg1UEMMRg+xg4h/5fZ8Aq
DgxYq3k8OkItf5SKrqumXb9Fx2v0Pw+n/Wa1WR9Oi1Vzag+bkVRQ8oNAMeDIDKGYPhRd6zQRtncY9orkeDi1zW6xP7Tb1XaMEaZ4
HuxIDWAg92H4GnlmTmQSIpkBHRI0YIaEIPaQEM9HMp4YgBROAwFhWJio+sFdjXbmaUxkJduQlUSO7FAVqBecXS1JPu/N/7icV839
4GRCcLDK6fEE0FISc/UwtyKMvH1IANZmXJ60rtnPlYYcfaJQlqT8EAT/TASola4VFwNB3gcmLIK7KUrTAwEk8lBSGzQHGEZr44cw
dB+Gq9nOVcssHFKdgQ4y1o91TcHhpT5/kY7Pn75Wf7XtsX542O3/fov+cG5sRYKyi87SDLvivJ6EkzxfguqeeBOeYeZpnDenU9P8
uOwWx+3L/bx0HASKHWAEgoqEAwBaWtuDgV1yFoOBXdcWGDqNQmM5FOgjGd5No6AcJzP7SAyGzJAEcQkcohZzcTTtpdltfzSL58P6
sjvcRyJJGQrkKlnXHIWBbKHJATJSquM9BK2PJgLF2A6FYkCDrzWXpAHRvmr3NA3iITJayP1mliVYyoeeqrW64YP0EIY2JfnQHHWT
IcFHlnTNdPzIsaOyvellYG4g2KK58Tp32AQXTOIw3n/w8y7opmM/QQJLQkBJEkjFhBDhmiZBrKZ7f5PdOcww8tBNH0fqB/fzR568
ssAAw6mEWmo5Dl8UyNXmJYDYHCAjbnNCtjnuA7r2deOrsB/dz+/lWTTINB7OAyfPA8Vx16yLlkf0mc77BBBbQ8kMJWBtMZwNqOla
FTCinGXNhdRJV7WYYAW4Jv3+WQrsVLScrMbWAiE4+7L9I9aqUgkWbMnxFL0ObYykp/uxRoLdLKJd0UrRKrKRyAksLqGvLQVtAoip
afZKNXeDpWxghcTy+LFVCtakakVF9dRjHJcTqSpACAvaYHxttFozDl3wAIWeP7Bm6le0XteN7xgdZv5+bcL5YZRPYvRDFsD1grPK
cVz/JSkwrpsdTLPApSsWILYTTkiHFkbs/+G+NCSBIBS+DSAfs4RRRsYxa4zd7rNsrbDFOEFPckLd9RhQWY9uMopWgFDOTmHkcKbu
A+LtDCp7M7D1bmeoWwND2WLhoF0WzDQN3ZVZ0WIhgpgYifMAXSOVlY9ggJzV00BQqrasoJooqDpRKgg12cJXZz6uWLS6VY/etWZ3
WTD/9irTeKg4WGOCEpdDyX3jkb7Bce7NItvhDTP4HobO/JS1Ha9TrfPTZIj/oaLzG8aGa1FNAxELwkX3Lhgv3xNenTItyIhXzzGm
IGNDUHdP1oztq6lbfGQQMnMtSs6EVist1w3/GQP9h38BruHAMg==""",
    "br:dia-57/fw-serraazul.log": """
eNqtmdtqHEcQhu/9FPsCu9S5uwYcCCL3vvALyKAQgxKHSEb47V2zjhJ1zbKlaQv2Slrpmzr9dRgCsiPCEfQAbQGIz+H3p+PD7eHX
m5vfPnw8fLz5cEA4EZwEToqLKroejr8c0OkUPz8hLiJ8+Ofr/d372/v7L0/HPx4f/344fvn6ePj07fHu4b00g/6ORpLF5xqJFhUw
f0lir0gKBpZICAvSNRIv2tlsILWK1AgcMkkK79Ei3tD3kRRFs/cofM7XSLqoSzzMS5JVJOPwQybFf7pqU1tJNNhEUnqvk3km+UJS
xIm740CiikTNuSUSS00K5402QUmi+EIm9YVa4T1mkKGeuCJxY8REkr4AFjaZYtsXJwKRHKfQAKk0oiPyvsolB8/eiywWLzQC0Ics
lzJOJj3lXrhOF5CCZM6DRlBZT82wtUSK6JIWGdFgrNyowcomR7RM6ou0QiO4oe3T8mYdKJM8Er2wCQH7D1KPDDkhrL/EpUOGvWT5
BZJgSSJaSQQcNkUinlqpseJukFihsaAli3EvqwnjFqVUo2w3itrGKi664RklsNuD0aU2iRGJCyUMn5P9v8TAuojRXTLMCljkO7kM
Wku1C5WkZ5JHf6jNogmzumbBiGZ1XQQjdwy1D3JbCkYodGpWZ5JaaRbifrOIeo4WV5Uc/cpBRnWqFZeUL4BaaRVN5CCB5uqSeGq9
Hixx9Z1WIUrLkiERLKzN0hmzSDMsOmSvgkVOQ2mVY4yRWE4L8XrcDLN8wizeyK6GD72EMc3AMIvGeVAuR4w2+BDL8ZYjxFkx1F/R
TZCnfAhjeXmMTYuUXTLyd0Y0iDJs3eYqGLcJGGN0L4TMC+2oPakyIfSNNcFie2SttkcT2rf/mBFsQVKnvbaZrpzm3YC1coaPnRh0
nyIKcOuZFGlfTqFoOCX0nmD0Kul4PirsS3vMaR+TDVe3Eupt5w7OrJLNYliAa7P8Tao5YEzFZMO9DQs/l6AOlnOQ8eJk+P946+v8
3exHHZ/xemHhf7r79Oft5/vj57+eaws3tRUo1AoVje9tUJf28AElTd8ERRdXoeRA+nc0fD2qd9kke8yu9dqFbUqdNGsuW3016a0P
2/i6VxeVZeJZneLxoFanPtFKom1hhr1KnRxn+pZlH2pYVg7YNLPkEUvfwgiLJS9WNt43icYcuun9+po8jAqYCtgwReF6dF+Qa9jU
yNYhw7hYX3WxGIZk3/GYYiXfkDS6cmkW8s8vyuuRPz51HqLPCMeY9CtJF6oDRvzzl42AhZJSqR3EM5MNjofkFYYLt+o43o13vsQg
aTlgpPVxPMyaGjbGy+EK64VZeL7ZDGZhebVmZ8tJv441tXDIzGgIZhmmxQx1humM0jtn2PpHUMNmKqxzzsOyh60vCJVo3zmPQTmX
l3ph1vrqBNpwYSOvTTKFd98BEId8CA==""",
    "br:dia-57/sa-cx-07-security.csv": """
eNqdklFLwzAUhd/9FaNPCulI0naWvI1twsCNYTtEyUtorzXYNiXJXPXXm07RdTKdQh7CzZecc25uKiuYaBAW8rXN0OwZajufolTo
AuzagF6KCtC1KlSdvjSA5s04zzUYg5awXWmVue0OWQntru5VziimI59gH0cpwSwiDNN7FI5oiJK7JJ0tUIR85HluHaKUEfyBZu0w
c+9qhS8RPc4HpOPjuMcj5E0Yv5V1rraGQ9uUSoMeQgte7yR5MRaqgPKNyytraXfIoUjAQnxUxOUutKgGV7IEMzhv49EFX8hMK6Me
LJ/lBfBx05QyE1aqmlcGXOm7k57HngHCaPhTyu6rDN+rd3pTYQW/UaKSdcEXydjpCl/YjSjlq8iV/rOBMPy1zZ/NNNmjFebJfIn8
32PPScBIxMj7LEUnDMiOp6cPlPtoynDHxxgfJt3RbzPg96g=""",
    "br:dia-57/sa-cx-07-mft.csv": """
eNqNkM1qhDAUhffzFPMAMdwbM/7tpNPuZhgq3QnlVlMb0ESiUuvT159FocgwIYsDB77zJc+mdz/XoflQjt3IKdPfqP9iL7pWV2rU
GjI9KfbkFPWqhBHhL/twQAGxj4znb51yXV6MvJghzkKYn+23qS2VHbvYcqjte6aqwZEpaE7OUToNNZ90yyRCICQTIAIPwYPTETER
foKSC4Er/063Kki5q5C27Zl6yl8tNdpU7JKlDP7DZCIiHgPOPLzXbUOnR4byZajpyKN+oFpPVFrH1aiYH8UzZVlBD4QH/hEhgeVy
2M6OQczl/EUQhZtB8LBBYc2nrrg2moVS7pIDmB/n3+0Ov7/omTQ=""",
    "br:dia-57/webmail-sa.jsonl": """
eNrFk00LwjAMhu/+itGzC0m7KvZ3ePIis0QdzE3aVQbif3cdeFD8RMaub0KevPk4i8YLkwiJcpYSpqiXhEYrg3IlpongE1dNTMhD
s4ey3hVVlINnF1Xbgu0SXI3zKHtn18UxBghBImQIug849qHs6/hgLXsvLpPzc3RmMvoGvWN3Y9MrtvyFLY2if21LVIBApGBB9+xt
XpTB8Ts26fHYGY7FlgaH8v1x38roB9+Hrt1N3YILJa+t47zhQZvoHk0bHOrg3w+/Z9Ni8Ge7Aj6RNWk=""",
}

REGISTRO_DE_CARGA = []   # (arquivo, origem, sha256): vai para as suas notas de método


def carregar(chave):
    """chave = 'lab:logs/firewall.log' ou 'br:dia-57/proxy-sa.log'. Devolve o texto."""
    prefixo, rel = chave.split(":", 1)                    # separa de onde vem e o caminho
    url = (RAW_LAB if prefixo == "lab" else RAW_BR) + rel  # monta a URL raw
    reserva = zlib.decompress(base64.b64decode(EMBUTIDOS[chave].replace("\n", "")))  # bytes da cópia embutida
    try:
        with urllib.request.urlopen(url, timeout=10) as r:  # tenta a rede, com tempo limite
            dados, origem = r.read(), "rede"
    except Exception:                                       # sem rede, 404, timeout: tanto faz
        dados, origem = reserva, "embutido"
    if origem == "rede" and dados != reserva:              # a versão online mudou desde que copiamos?
        print(f"⚠️  {rel}: versão online difere da embutida; usando a embutida (números do dia batem com ela)")
        dados, origem = reserva, "embutido (online divergente)"
    sha = hashlib.sha256(dados).hexdigest()                # impressão digital do que vamos analisar
    REGISTRO_DE_CARGA.append((rel, origem, sha))           # anota para o relatório
    return dados.decode("utf-8")


print(f"{len(EMBUTIDOS)} arquivos disponíveis. Use carregar('...').")

In [ ]:
import csv, io, json, re                                  # formatos de texto
from datetime import datetime, timedelta, timezone        # tempo, sempre com fuso

UTC = timezone.utc
try:
    from zoneinfo import ZoneInfo                         # banco IANA (stdlib, Python 3.9+)
    NY, SP = ZoneInfo("America/New_York"), ZoneInfo("America/Sao_Paulo")
except Exception:                                         # sem tzdata: fusos fixos válidos nas datas do caso
    NY, SP = timezone(timedelta(hours=-4)), timezone(timedelta(hours=-3))

FW = carregar("lab:logs/firewall.log").splitlines()
DOC = carregar("lab:logs/docportal-app.jsonl").splitlines()
BAST = carregar("lab:logs/bastion01-auth.log").splitlines()
FS01 = carregar("lab:logs/fs01-auth.log").splitlines()
print(len(FW), len(DOC), len(BAST), len(FS01), "linhas")

In [ ]:
# Tabela de carga: o que foi lido, de onde e com qual hash.
for rel, origem, sha in REGISTRO_DE_CARGA:     # uma linha por arquivo
    print(f"{rel:40s} {origem:10s} {sha[:16]}…")  # 16 primeiros caracteres bastam para conferir de olho

## 2. A linha do tempo "fw+app" corrigida (passo 2 do original)

Cada linha vira `message`, `datetime`, `timestamp_desc`, mais `host`, `evidence_ref` e **`time_correction`**, a correção escrita em todas as linhas.

In [ ]:
DESVIO_FS01 = timedelta(seconds=83)                       # medido no Dia 56 pela porta 41766

eventos = []                                              # cada evento: dict com as colunas do Timesketch

for n, linha in enumerate(FW, start=1):                   # firewall: hora local de Nova York, sem fuso
    data, hora = linha.split()[:2]
    local = datetime.fromisoformat(f"{data} {hora}").replace(tzinfo=NY)
    eventos.append({"message": linha, "datetime": local.astimezone(UTC), "host": "fw01",
                    "timestamp_desc": "Connection Allowed" if " ACCEPT " in linha else "Connection Dropped",
                    "evidence_ref": f"firewall.log:{n}",
                    "time_correction": "America/New_York to UTC (UTC-4, measured on port pairs)",
                    "timeline": "EVID-001 fw+app (corrected CSV)"})

for n, linha in enumerate(DOC, start=1):                  # portal: JSON com o relógio do fs01 (83 s adiantado)
    r = json.loads(linha)
    dt = datetime.fromisoformat(r["ts"].replace("Z", "+00:00")) - DESVIO_FS01
    msg = f'{r["event"]} user={r["user"]} src_ip={r.get("src_ip","-")} path={r.get("path","-")} count={r.get("count","-")}'
    eventos.append({"message": msg, "datetime": dt, "host": "fs01/docportal", "timestamp_desc": "Event Recorded",
                    "evidence_ref": f"docportal-app.jsonl:{n}",
                    "time_correction": "fs01 clock -83 s (measured on port 41766)",
                    "timeline": "EVID-001 fw+app (corrected CSV)"})


def exportar_ts_csv(evs, caminho):
    """CSV no formato de importação do Timesketch: message, datetime (ISO 8601), timestamp_desc + extras."""
    with open(caminho, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["message", "datetime", "timestamp_desc", "host", "evidence_ref", "time_correction"])
        for e in sorted(evs, key=lambda e: e["datetime"]):
            w.writerow([e["message"], e["datetime"].isoformat(), e["timestamp_desc"],
                        e["host"], e["evidence_ref"], e["time_correction"]])

exportar_ts_csv(eventos, "lab-p4-fw-app.ts.csv")
print(open("lab-p4-fw-app.ts.csv", encoding="utf-8").read().splitlines()[:3])

In [ ]:
# A linha do tempo "EVID-001 syslog (plaso)": como o Plaso a deixaria, SEM corrigir o fs01.
SYSLOG = re.compile(r"^(\w{3})\s+(\d{1,2}) (\d\d:\d\d:\d\d) (\S+) (.*)$")
for nome, linhas in (("bastion01-auth.log", BAST), ("fs01-auth.log", FS01)):
    for n, linha in enumerate(linhas, start=1):
        mes, dia, hora, host, resto = SYSLOG.match(linha).groups()
        dt = datetime.strptime(f"2026 {mes} {dia} {hora}", "%Y %b %d %H:%M:%S").replace(tzinfo=UTC)
        eventos.append({"message": resto, "datetime": dt, "host": host, "timestamp_desc": "Entry Written",
                        "evidence_ref": f"{nome}:{n}", "time_correction": "",
                        "timeline": "EVID-001 syslog (plaso)"})
print(f"{len(eventos)} eventos no sketch 'LAB-P4 (synthetic)' (sem o EVID-004, que não temos aqui)")

## 3. Mini-Timesketch: busca, *tags*, buscas salvas, comentários

A nossa busca é simplificada: todos os termos entre aspas precisam aparecer (`AND`), ou qualquer um deles (`OR`).

In [ ]:
TAGS, COMENTARIOS, BUSCAS_SALVAS = {}, {}, {}             # o "estado" do sketch

def buscar(*termos, modo="AND", evs=None):
    """Devolve eventos cuja message (+ evidence_ref) contém os termos."""
    evs = eventos if evs is None else evs
    testar = all if modo == "AND" else any
    return sorted([e for e in evs if testar(t in e["message"] for t in termos)], key=lambda e: e["datetime"])

def marcar(evs, tag):
    for e in evs:
        TAGS.setdefault(e["evidence_ref"], set()).add(tag)  # a tag fica no evento, pela referência

def salvar_busca(nome, *termos, modo="AND"):
    BUSCAS_SALVAS[nome] = (termos, modo)                    # guarda a consulta, não o resultado
    return buscar(*termos, modo=modo)

def hora(e):
    return e["datetime"].strftime("%H:%M:%SZ")

spray = salvar_busca("initial-access", "Failed password", "203.0.113.45")
marcar([spray[0], spray[-1]], "initial-access")            # só o primeiro e o último, como no original
login = buscar("Accepted password for svc_backup"); marcar(login, "initial-access")
lateral = salvar_busca("lateral", "41766"); marcar(lateral, "lateral")
staging = salvar_busca("staging", "bulk_export", "tar -czf", modo="OR"); marcar(staging, "staging")
exfil = salvar_busca("exfil", "198.51.100.23"); marcar(exfil, "exfil")

print(f"spray: {len(spray)} falhas, de {hora(spray[0])} a {hora(spray[-1])}")
for nome, evs in (("lateral", lateral), ("staging", staging), ("exfil", exfil)):
    print(f"\n[{nome}]")
    for e in evs:
        print(f"  {hora(e)}  {e['timeline'][:24]:24s} {e['evidence_ref']:24s} {e['message'][:60]}")

Repare no `lateral`: o evento do `fs01` aparece às **03:20:45Z** (linha do tempo do Plaso, sem correção) e o do firewall às **03:19:21Z**. O Timesketch não deixa editar; você **comenta**.

In [ ]:
# Ligações automáticas: valores (IPs e portas) que aparecem em FONTES DIFERENTES.
from collections import defaultdict
CAMPO = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b|port (\d{4,5})|:(\d{4,5})\b")
indice = defaultdict(set)                                  # valor -> {arquivos de origem}
for e in eventos:
    arquivo = e["evidence_ref"].split(":")[0]
    for m in CAMPO.finditer(e["message"]):
        indice[m.group(1) or m.group(2) or m.group(0)].add(arquivo)
pontes = {v: sorted(f) for v, f in indice.items() if len(f) > 1 and not v.startswith("10.10.40")}
for v in ("41766", "198.51.100.23", "203.0.113.45", "10.10.20.5"):
    fontes = sorted(indice.get(v, []))
    marca = "🔗 ponte entre fontes" if len(fontes) > 1 else "uma fonte só: aqui não há ligação a fazer"
    print(f"{v:15s} {fontes}  {marca}")
# 198.51.100.23 só aparece no firewall NESTE sketch. A ponte com a WS-FIN-07 vem da memória (Dia 54),
# que é outra evidência: por isso a história a marca como inferida.

In [ ]:
# Comentários de ligação (passo 6 do original): evento -> outro evento + campo compartilhado.
def comentar(ev, outro, campo, corrigido=None):
    txt = (f"Mesma conexão/objeto que {outro['host']} em {hora(outro)} "
           f"({outro['timeline']}, {outro['evidence_ref']}); campo compartilhado: {campo}.")
    if corrigido:
        txt += f" Hora corrigida {corrigido}."
    COMENTARIOS.setdefault(ev["evidence_ref"], []).append(txt)

fw_41766 = next(e for e in lateral if e["host"] == "fw01")
fs_41766 = next(e for e in lateral if e["host"] == "fs01")
comentar(fs_41766, fw_41766, "porta de origem 41766",
         corrigido=(fs_41766["datetime"] - DESVIO_FS01).strftime("%H:%M:%SZ"))
comentar(fw_41766, fs_41766, "porta de origem 41766")
tar = next(e for e in staging if "tar -czf" in e["message"])
bulk = next(e for e in staging if "bulk_export" in e["message"])
comentar(tar, bulk, "caminho finance/2026-Q1 e usuário svc_backup",
         corrigido=(tar["datetime"] - DESVIO_FS01).strftime("%H:%M:%SZ"))
envio = next(e for e in exfil if "48213904" in e["message"])
comentar(envio, bulk, "nenhum campo: só tamanho (48.006.112 → 48.213.904) e sequência — INFERIDO")
comentar(login[0], fw_41766, "host bastion01 10.10.20.5 e usuário svc_backup (sessão aberta às 03:14:08)")

for ref, cs in COMENTARIOS.items():
    for c in cs:
        print(f"💬 {ref}: {c}")
print(f"\n{sum(len(c) for c in COMENTARIOS.values())} comentários entre linhas do tempo (meta: 4+)")

## 4. A história `LAB-P4 intrusion path` (Markdown)

In [ ]:
HISTORIA_LAB = f"""# LAB-P4 intrusion path (synthetic)

## 1. Correções de tempo
- fw01: America/New_York (UTC-4) → UTC, medido em pares de porta (ex.: 41766).
- fs01 e docportal: −83 s, medido na porta 41766 (fw01 03:19:21Z × fs01 03:20:45 no relógio do fs01, 03:19:22Z corrigido; o sshd registra o aceite 1 s depois do SYN do firewall).
- Os eventos do fs01 na linha do tempo do Plaso NÃO estão corrigidos: veja os comentários.

## 2. Password spray contra o bastion01
{len(spray)} falhas de {hora(spray[0])} a {hora(spray[-1])}, {len({re.search(r"for (?:invalid user )?(\S+)", e["message"]).group(1) for e in spray})} usuários, origem única 203.0.113.45. Busca salva: `initial-access`.

## 3. Login por senha como svc_backup às {hora(login[0])}
Fonte: {login[0]["evidence_ref"]}.

## 4. SSH bastion01 → fs01, 03:19:21Z a 03:19:22Z
Ligado pela **porta de origem 41766** ({fw_41766["evidence_ref"]} ↔ {fs_41766["evidence_ref"]}). Busca salva: `lateral`.

## 5. Exportação e arquivamento de finance/2026-Q1
bulk_export às {(bulk["datetime"]).strftime("%H:%M:%SZ")} ({bulk["evidence_ref"]}); tar às {(tar["datetime"] - DESVIO_FS01).strftime("%H:%M:%SZ")} corrigido ({tar["evidence_ref"]}). Ligados por **caminho e usuário**. Busca salva: `staging`.

## 6. Saída de 48.213.904 bytes para 198.51.100.23:443 às {hora(envio)}
({envio["evidence_ref"]}). **Inferido**: a ligação com o arquivo exportado é só por tamanho e sequência. O mesmo IP aparece na memória da WS-FIN-07 (Dia 54): **inferido**, só o IP é compartilhado. Busca salva: `exfil`.

## 7. Questões em aberto
- Como a senha do svc_backup era conhecida?
- O que explicam as falhas das 02:10 contra a WS-FIN-07 a partir do bastion01?
- A imagem USB (EVID-004) tem alguma ligação com esta sequência?
"""
print(HISTORIA_LAB)

## 5. ➕ A história "Isca — SA-CX-07"

Agora o caso brasileiro. Normalizamos as cinco fontes, exportamos `isca-sa-cx-07.ts.csv` e deixamos o índice de campos procurar as pontes.

In [ ]:
PROXY = re.compile(r"^(\w{3})\s+(\d{1,2}) (\d\d:\d\d:\d\d) proxy-sa \S+ (.*)$")
isca = []                                                 # sketch separado: nunca misture clientes

for n, l in enumerate(carregar("br:dia-57/proxy-sa.log").splitlines(), 1):
    mes, dia, hh, resto = PROXY.match(l).groups()
    dt = datetime.strptime(f"2026 {mes} {dia} {hh}", "%Y %b %d %H:%M:%S").replace(tzinfo=UTC)
    isca.append({"message": resto, "datetime": dt, "host": "proxy-sa", "timestamp_desc": "Request Logged",
                 "evidence_ref": f"proxy-sa.log:{n}", "time_correction": "ano 2026 confirmado; relógio já em UTC"})
for n, l in enumerate(carregar("br:dia-57/fw-serraazul.log").splitlines(), 1):
    d, h = l.split()[:2]
    dt = datetime.fromisoformat(f"{d} {h}").replace(tzinfo=SP).astimezone(UTC)
    isca.append({"message": l[20:], "datetime": dt, "host": "fw-sa",
                 "timestamp_desc": "Connection Allowed" if " ACCEPT " in l else "Connection Dropped",
                 "evidence_ref": f"fw-serraazul.log:{n}",
                 "time_correction": "America/Sao_Paulo → UTC (+3 h), medido no par proxy/firewall de lnk.example 11:19:38Z"})
for n, r in enumerate(csv.DictReader(io.StringIO(carregar("br:dia-57/sa-cx-07-security.csv"))), 2):
    msg = f'EventID {r["EventID"]} user={r["TargetUserName"]} {r["NewProcessName"]} parent={r["ParentProcessName"]}'
    isca.append({"message": msg, "datetime": datetime.fromisoformat(r["TimeCreatedUtc"].replace("Z", "+00:00")),
                 "host": "SA-CX-07", "timestamp_desc": "Event Recorded",
                 "evidence_ref": f"sa-cx-07-security.csv:{n}", "time_correction": ""})
for n, r in enumerate(csv.DictReader(io.StringIO(carregar("br:dia-57/sa-cx-07-mft.csv"))), 2):
    dt = datetime.fromisoformat(r["Created0x30"][:19]).replace(tzinfo=UTC)   # $FN criado (o $SI pode mentir)
    isca.append({"message": f'{r["ParentPath"]}\\{r["FileName"]} size={r["FileSize"]} SI_created={r["Created0x10"][:19]}',
                 "datetime": dt, "host": "SA-CX-07", "timestamp_desc": "FN Creation Time",
                 "evidence_ref": f"sa-cx-07-mft.csv:{n}", "time_correction": ""})
for n, l in enumerate(carregar("br:dia-57/webmail-sa.jsonl").splitlines(), 1):
    r = json.loads(l)
    isca.append({"message": f'{r["event"]} user={r["user"]} src_ip={r["src_ip"]} result={r["result"]}',
                 "datetime": datetime.fromisoformat(r["ts"].replace("Z", "+00:00")), "host": "webmail-sa",
                 "timestamp_desc": "Event Recorded", "evidence_ref": f"webmail-sa.jsonl:{n}", "time_correction": ""})

exportar_ts_csv(isca, "isca-sa-cx-07.ts.csv")
print(f"isca-sa-cx-07.ts.csv: {len(isca)} eventos")

In [ ]:
# Buscas da Prática BR e o índice de pontes.
def mostrar(nome, evs):
    print(f"\n[{nome}] {len(evs)} eventos")
    for e in evs[:6]:
        print(f"  {hora(e)} {e['host']:10s} {e['evidence_ref']:26s} {e['message'][:58]}")

mostrar("initial-access", buscar("lnk.example", "serraazul-pix.example", modo="OR", evs=isca))
mostrar("execution", buscar("msa-atualizador.exe", "Modulo_Seguranca_SerraAzul.zip", modo="OR", evs=isca))
c2 = buscar("198.51.100.140", evs=isca)
mostrar("c2-suspeito", c2)
mostrar("conta-webmail", buscar("203.0.113.91", evs=isca))

# Intervalos entre contatos ao C2 (só no proxy, para não contar duas vezes)
t = [e["datetime"] for e in c2 if e["host"] == "proxy-sa"]
gaps = [(b - a).total_seconds() for a, b in zip(t, t[1:])]
print(f"\nIntervalos proxy→198.51.100.140: min {min(gaps):.0f} s, max {max(gaps):.0f} s, n={len(gaps)}")

### 🧩 Exercício: o tamanho do intervalo

O `min` saiu muito menor que 300 s? Olhe as linhas: uma delas é o **envio de 3 MB às 12:07:10**, que não faz parte do ritmo. É o tipo de coisa que você só vê olhando o dado, não a média. Você vai medir a dispersão a sério no Dia 59.

In [ ]:
HISTORIA_ISCA = """# Isca — SA-CX-07 (sintético)

## 1. Correções de tempo
- fw-serraazul.log: America/Sao_Paulo → UTC (+3 h), medido no par proxy/firewall do GET lnk.example/sa7 (11:19:38Z nos dois).
- proxy-sa.log: já em UTC; ano 2026 confirmado (cópia de 6/out/2026).

## 2. Clique — 11:19:38Z
GET http://lnk.example/sa7 (302) e CONNECT serraazul-pix.example:443 às 11:19:41Z. Ligação: IP de origem 10.20.40.57 + usuário cx.centro07 (proxy).

## 3. Download — 11:23:12Z / 11:23:14Z
~412 KB do site falso; Modulo_Seguranca_SerraAzul.zip (410.624 bytes) em Downloads. **Inferido** por horário e tamanho.

## 4. Execução — 11:24:29Z a 11:24:44Z
msa-atualizador.exe criado ($FN 2026; $SI diz 2021), processo às 11:24:31Z, schtasks.exe filho às 11:24:44Z. Ligação: caminho do arquivo.

## 5. Contato periódico — desde 11:25:02Z
atualiza-modulo.example (198.51.100.140) a cada ~300 s; 3.145.728 bytes enviados às 12:07:10Z. Conteúdo **não observado**.

## 6. Conta de webmail — 12:31Z a 12:33Z
cx.centro07: 3 falhas e 1 sucesso a partir de 203.0.113.91; regra de caixa postal criada. Ligação com o notebook: **inferida**, só pelo nome da conta.

## 7. Questões em aberto
- A caixa digitou credenciais no site falso? Nenhuma fonte entregue mostra o formulário.
- O que havia nos 3 MB?
- O que a regra de caixa postal faz?
- Outros notebooks da agência contataram 198.51.100.140?
"""
print(HISTORIA_ISCA)

## 6. 🎮 Quiz: seta cheia ou tracejada?

Checkpoint BR: pelo menos 5 de 6.

In [ ]:
# ------------------------------------------------------------
# QUIZ: função genérica de múltipla escolha
# MODO_JOGO = True  -> pergunta e conta acertos
# MODO_JOGO = False -> só mostra gabarito e explicação
# ------------------------------------------------------------
def jogar_quiz(perguntas, modo_jogo=True, minimo=None):
    acertos = 0                                            # placar
    for n, (enunciado, opcoes, certa, porque) in enumerate(perguntas, start=1):
        print(f"\n[{n}/{len(perguntas)}] {enunciado}")    # o enunciado
        for letra, texto in opcoes.items():                # as alternativas a), b), c)
            print(f"   {letra}) {texto}")
        if modo_jogo:
            resposta = ""
            for _ in range(5):                             # até 5 tentativas de digitar uma letra válida
                resposta = input("   Sua resposta: ").strip().lower()
                if resposta in opcoes:
                    break
            ok = (resposta == certa)                       # acertou?
            acertos += ok
            print("   ✅ Acertou!" if ok else f"   ❌ Era {certa}).")
        else:
            print(f"   Gabarito: {certa})")
        print(f"   Por quê: {porque}")
    if modo_jogo:
        print(f"\n🏁 Resultado: {acertos}/{len(perguntas)}")
        if minimo is not None:
            print("Checkpoint BR cumprido. 🥷" if acertos >= minimo
                  else f"Checkpoint BR pede pelo menos {minimo}. Releia o README e tente de novo.")
    return acertos

In [ ]:
MODO_JOGO = True   # False = só gabarito

PERGUNTAS = [
    ("fw01 ACCEPT 10.10.20.5:41766 → fs01:22 e o 'Accepted publickey ... port 41766' do fs01:",
     {"a": "seta cheia (porta de origem compartilhada)", "b": "seta tracejada", "c": "não há ligação"},
     "a", "O campo compartilhado é nomeável e único na janela: porta 41766."),
    ("bulk_export de 48.006.112 bytes e a conexão de 48.213.904 bytes para 198.51.100.23:",
     {"a": "seta cheia (os tamanhos batem)", "b": "seta tracejada (só tamanho e sequência)", "c": "não há ligação"},
     "b", "Tamanho parecido não é campo compartilhado; o firewall não vê conteúdo."),
    ("Login externo no webmail como cx.centro07 e o notebook SA-CX-07:",
     {"a": "seta cheia, a conta é a mesma", "b": "seta tracejada, liga só pelo nome da conta", "c": "prova que o malware roubou a senha"},
     "b", "A senha pode ter vindo do formulário do site falso, de reuso, de outro lugar. Origem em aberto."),
    ("GET lnk.example/sa7 no proxy às 11:19:38Z e a linha do firewall 08:19:38 para 198.51.100.201:",
     {"a": "seta cheia: mesmo IP de origem, mesmo destino, mesmo segundo após a correção", "b": "tracejada", "c": "fontes diferentes nunca se ligam"},
     "a", "IP de origem + destino + horário corrigido. E é essa ligação que mede a correção."),
    ("O fs01 está 83 s adiantado no Timesketch. O certo é:",
     {"a": "editar o timestamp no banco do Timesketch", "b": "nota na história + comentário com a hora corrigida em cada evento citado", "c": "ignorar, 83 s não mudam nada"},
     "b", "O dado original fica intacto (cadeia de custódia); a correção fica registrada ao lado."),
    ("Sua história do SA-CX-07 ficou sem nenhuma questão em aberto. Isso indica:",
     {"a": "trabalho completo", "b": "que você concluiu mais do que os dados mostram", "c": "que faltam fontes no sketch"},
     "b", "É o critério do próprio checkpoint do original."),
]
jogar_quiz(PERGUNTAS, MODO_JOGO, minimo=5)

## 7. 💾 Levando os artefatos

In [ ]:
from pathlib import Path
Path("notas").mkdir(exist_ok=True)
Path("notas/historia-lab-p4.md").write_text(HISTORIA_LAB, encoding="utf-8")
Path("notas/historia-isca-sa-cx-07.md").write_text(HISTORIA_ISCA, encoding="utf-8")
arquivos = ["lab-p4-fw-app.ts.csv", "isca-sa-cx-07.ts.csv", "notas/historia-lab-p4.md", "notas/historia-isca-sa-cx-07.md"]
print("💾", ", ".join(arquivos))
try:
    from google.colab import files
    for f in arquivos:
        files.download(f)
except ImportError:
    print("(Fora do Colab: os arquivos ficaram na pasta atual.)")

## ✅ Antes de fechar o Dia 58

- [ ] `lab-p4-fw-app.ts.csv` com `time_correction` em todas as linhas
- [ ] 5 buscas salvas e pelo menos 4 comentários entre linhas do tempo
- [ ] História do LAB-P4 com 7 seções e setas inferidas marcadas
- [ ] História "Isca — SA-CX-07" num sketch separado
- [ ] Quiz com pelo menos 5/6

**Amanhã (Dia 59):** forense de rede. O "a cada ~300 s" vira número: média, dispersão, e um grafo de fluxos.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>